# GCC-4P committed TRAIN
**THIS NOTEBOOK IS DESIGNED FOR: KAGGLE â†’ SAVE VERSION â†’ SAVE & RUN ALL**
It is unattended and train-only. Successful completion leaves verified top-level output in `/kaggle/working` for the committed Version.

In [ ]:
# 1 â€” isolate GPU 0 before torch import
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'

In [ ]:
# 2 â€” environment audit
import platform,subprocess,sys
subprocess.run(['nvidia-smi'],check=True)
import torch
assert torch.cuda.is_available(),'KAGGLE_CUDA_GPU_REQUIRED';assert torch.cuda.device_count()==1,'GCC4P_REQUIRES_EXACTLY_ONE_VISIBLE_GPU'
props=torch.cuda.get_device_properties(0);print({'execution_surface':'KAGGLE','gpu':props.name,'vram_bytes':props.total_memory,'cuda':torch.version.cuda,'torch':torch.__version__,'python':sys.version,'platform':platform.platform()})

In [ ]:
# 3 â€” pinned dependencies
import importlib.util
if importlib.util.find_spec('torchao'):subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.51.3','peft==0.20.0','accelerate','safetensors','psutil'],check=True)
os.environ['WANDB_DISABLED']='true';os.environ['HF_HUB_DISABLE_TELEMETRY']='1'

In [ ]:
# 4 â€” discover and verify exploded-or-ZIP canonical recovery input
import hashlib,json,shutil,zipfile
from pathlib import Path,PurePosixPath
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/gcc4p');EXPECTED_ZIP='7f2e026e71131ac4a1c6cf09c02066655fc40b213bbb9034dce673d8c5fc10b3';EXPECTED_CORPUS='sha256:b5db5a57c5e0f1cc041e5bd138cd53e06359f349a8d7f4a5ec36018c985d64ef'
REQUIRED=['SHA256SUMS.txt','corpus/manifest.json','corpus/oracle.audit.json','corpus/leakage.audit.json','corpus/train.jsonl','corpus/validation.jsonl','corpus/qualification.jsonl','corpus/holdout.jsonl','corpus/adversarial.jsonl','historical/validation.jsonl','historical/qualification.jsonl','historical/holdout.jsonl','historical/adversarial.jsonl','historical/gold.jsonl','ood/benchmark.jsonl','ood/manifest.json','scripts/train_targeted_student_v04.py','scripts/gcc4p_recovery.py','scripts/experiment.config.json','scripts/evaluate_interpretation_student_v04.py','scripts/lost-run.evidence.json','baselines/v02.evidence.review.json','baselines/v03.generalization.audit.json','policy/declared.role.fidelity.policy.json']
roots=[p for p in INPUT.rglob('SHA256SUMS.txt') if all((p.parent/r).is_file() for r in REQUIRED)]
if len(roots)>1:raise RuntimeError('MULTIPLE_GCC4P_EXPLODED_INPUTS')
if roots: source=roots[0].parent;input_mode='EXPLODED_KAGGLE_DATASET';outer_zip_digest='NOT_APPLICABLE_KAGGLE_EXPLODED_INPUT'
else:
    archives=list(INPUT.rglob('PA-INTERPRETATION-STUDENT-v0.4-GCC4P-RECOVERY-INPUT.zip'))
    if len(archives)!=1:raise RuntimeError('GCC4P_INPUT_NOT_FOUND' if not archives else 'MULTIPLE_GCC4P_ZIP_INPUTS')
    archive=archives[0];actual=hashlib.sha256(archive.read_bytes()).hexdigest();assert actual==EXPECTED_ZIP,'GCC4P_RECOVERY_INPUT_SHA256_MISMATCH'
    source=Path('/kaggle/working/gcc4p-source');shutil.rmtree(source,ignore_errors=True)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            path=PurePosixPath(name);assert not path.is_absolute() and '..' not in path.parts and '\\' not in name,'UNSAFE_ZIP_PATH'
        z.extractall(source)
    input_mode='RECOVERY_ZIP';outer_zip_digest=actual
verified=0
for line in (source/'SHA256SUMS.txt').read_text().splitlines():
    expected,relative=line.split('  ',1);path=PurePosixPath(relative);assert not path.is_absolute() and '..' not in path.parts and '\\' not in relative,'UNSAFE_GCC4P_PAYLOAD_PATH';payload=source/relative;assert payload.is_file(),relative;assert hashlib.sha256(payload.read_bytes()).hexdigest()==expected,relative;verified+=1
assert json.loads((source/'corpus/manifest.json').read_text())['aggregate_corpus_digest']==EXPECTED_CORPUS,'GCC4P_CORPUS_DIGEST_MISMATCH'
print({'input_mode':input_mode,'outer_zip_digest':outer_zip_digest,'payload_hashes_verified':verified})

In [ ]:
# 5 â€” materialize verified input without mutating /kaggle/input
shutil.rmtree(WORK,ignore_errors=True);shutil.copytree(source,WORK);assert all((WORK/r).is_file() for r in REQUIRED)

In [ ]:
# 6 â€” exact pinned-base and API preflight
import inspect,peft,transformers
from huggingface_hub import HfApi
from transformers import Trainer,TrainingArguments
assert transformers.__version__=='4.51.3';assert peft.__version__=='0.20.0';assert 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters
try:HfApi().model_info('Qwen/Qwen3-0.6B-Base',revision='da87bfb608c14b7cf20ba1ce41287e8de496c0cd')
except Exception as error:raise RuntimeError(f'KAGGLE_INTERNET_OR_PINNED_BASE_UNAVAILABLE: {error}') from error
subprocess.run([sys.executable,str(WORK/'scripts/train_targeted_student_v04.py'),'--help'],check=True)

In [ ]:
# 7 â€” committed-run persistence root and provenance
PERSIST_ROOT='/kaggle/working/PlannerAgent/GCC4P';os.environ['PLANNERAGENT_GCC4P_PERSIST_ROOT']=PERSIST_ROOT
candidate=Path(PERSIST_ROOT)/'PA-INTERPRETATION-STUDENT-v0.4';state=candidate/'state';state.mkdir(parents=True,exist_ok=True)
environment={'execution_surface':'KAGGLE','gpu':props.name,'vram_bytes':props.total_memory,'visible_gpu_count':torch.cuda.device_count(),'cuda':torch.version.cuda,'torch':torch.__version__,'python':sys.version,'transformers':transformers.__version__,'peft':peft.__version__};(state/'kaggle.environment.json').write_text(json.dumps(environment,sort_keys=True,indent=2)+'\n')

In [ ]:
# 8 â€” TRAIN ONLY
subprocess.run([sys.executable,str(WORK/'scripts/train_targeted_student_v04.py'),'--phase','train','--persist-root',PERSIST_ROOT],check=True)

In [ ]:
# 9 â€” verify COMPLETE, exact identity, manifest, and every adapter hash
sys.path.insert(0,str(WORK/'scripts'));from gcc4p_recovery import identity,sha_file,valid_final_adapter
config=json.loads((WORK/'scripts/experiment.config.json').read_text());corpus=json.loads((WORK/'corpus/manifest.json').read_text());expected_identity=identity(config,corpus['aggregate_corpus_digest']);final=candidate/'final-adapter';manifest=valid_final_adapter(final,expected_identity);assert manifest,'VERIFIED_FINAL_ADAPTER_REQUIRED'
adapter_zip=candidate/'result'/'PA-INTERPRETATION-STUDENT-v0.4-ADAPTER-GCC4P.zip';zip_manifest=json.loads((candidate/'result'/'adapter-zip.manifest.json').read_text());assert adapter_zip.is_file();assert sha_file(adapter_zip)==zip_manifest['sha256']

In [ ]:
# 10 â€” emit top-level committed outputs; this is the final cell
top_zip=Path('/kaggle/working/PA-INTERPRETATION-STUDENT-v0.4-ADAPTER-GCC4P.zip');shutil.copy2(adapter_zip,top_zip);assert sha_file(top_zip)==zip_manifest['sha256']
training_complete=json.loads((candidate/'state/training-complete.json').read_text());run_manifest={**expected_identity,'training_complete':True,'persistent_adapter_verified':True,'execution_surface':'KAGGLE','adapter_zip_sha256':sha_file(top_zip),'adapter_artifact_digest':manifest['adapter_digest'],'environment_evidence':environment,'training_metrics':training_complete,'timestamp_evidence':training_complete['completed_at'],'input_mode':input_mode,'payload_hashes_verified':verified}
run_path=Path('/kaggle/working/PA-INTERPRETATION-STUDENT-v0.4-TRAIN-RUN.json');run_path.write_text(json.dumps(run_manifest,sort_keys=True,indent=2)+'\n');Path('/kaggle/working/PA-INTERPRETATION-STUDENT-v0.4-TRAIN-SHA256SUMS.txt').write_text(f'{sha_file(top_zip)}  {top_zip.name}\n{sha_file(run_path)}  {run_path.name}\n')
assert top_zip.is_file() and run_path.is_file();print('KAGGLE_COMMITTED_V04_TRAIN_OUTPUT_READY');print({'adapter_path':str(top_zip),'adapter_size':top_zip.stat().st_size,'adapter_sha256':sha_file(top_zip),'train_run_manifest':str(run_path)})